# 05 — Final Load Prep & KPI Computation
**Project:** Swiggy India Restaurant Analytics
**Goal:** Compute all KPIs, add segmentation columns, and export the Tableau-ready dataset.

In [1]:
%pip install pandas numpy 


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip3.13 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().resolve().name == 'notebooks'
    else Path.cwd().resolve()
)

DATA_PATH = PROJECT_ROOT / 'data/processed/cleaned_dataset.csv'
TABLEAU_PATH = PROJECT_ROOT / 'data/processed/tableau_ready_dataset.csv'

df = pd.read_csv(DATA_PATH)
print('Input:', df.shape)
df.head()

Input: (10083, 12)


,restaurant_name,cuisine,rating,offer_name,area,pure_veg,location,price_num,ratings_count,primary_cuisine,is_veg,num_offers
0,Burger King,"Burgers, American",4.2,DEAL OF DAY\n60% OFF UPTO ₹120\nUSE STEALDEALA...,Basavanagudi,No,Bangalore,350.0,10000.0,Burgers,0,3
1,Pizza Hut,Pizzas,3.9,"50% OFF UPTO ₹100\nUSE SWIGGYITABOVE ₹189, FLA...",Basavanagudi,No,Bangalore,350.0,5000.0,Pizzas,0,2
2,Theobroma,"Desserts, Bakery",4.6,"40% OFF UPTO ₹80\nUSE SPECIALSON SELECT ITEMS,...",Ashok Nagar,No,Bangalore,400.0,1000.0,Desserts,0,2
3,Starbucks Coffee,"Beverages, Cafe",4.2,"40% OFF UPTO ₹80\nUSE TRYNEWON SELECT ITEMS, F...",Basaveshwaranagar,No,Bangalore,400.0,100.0,Beverages,0,3
4,KFC,"Burgers, Biryani",4.2,No Offer,Basavanagudi,No,Bangalore,400.0,10000.0,Burgers,0,0


---
## KPI 1 — Average Rating by Primary Cuisine

In [3]:
df['avg_rating_by_cuisine'] = df.groupby('primary_cuisine')['rating'].transform('mean')

cuisine_kpi = (
    df.groupby('primary_cuisine')
    .agg(
        avg_rating=('rating', 'mean'),
        restaurant_count=('restaurant_name', 'count'),
        rated_count=('rating', 'count')
    )
    .round(3)
    .sort_values('avg_rating', ascending=False)
)
cuisine_kpi['pct_rated'] = (cuisine_kpi['rated_count'] / cuisine_kpi['restaurant_count'] * 100).round(1)

print("Top 15 Cuisines by Average Rating:")
print(cuisine_kpi.head(15).to_string())

Top 15 Cuisines by Average Rating:
                    avg_rating  restaurant_count  rated_count  pct_rated
primary_cuisine                                                         
Telangana                4.700                 2            1       50.0
Sushi                    4.700                 1            1      100.0
Ice Cream Cakes          4.675                 5            4       80.0
Paan                     4.566                32           29       90.6
Cakes and Pastries       4.529                 9            7       77.8
North Eastern            4.500                 1            1      100.0
French                   4.450                 2            2      100.0
Rolls & Wraps            4.400                 9            6       66.7
Oriental                 4.400                 1            1      100.0
Parsi                    4.400                 1            1      100.0
Keto                     4.333                 3            3      100.0
Pan-Asian       

---
## KPI 2 — Value Index (Rating per Rs100 of Price)

In [4]:
df['value_index'] = np.where(
    df['price_num'] > 0,
    (df['rating'] / (df['price_num'] / 100)).round(4),
    np.nan
)

print("Value Index Summary:")
print(df['value_index'].describe().round(3))
print()
print("Top 10 highest value restaurants:")
top_value = (
    df[['restaurant_name', 'location', 'primary_cuisine', 'price_num', 'rating', 'value_index']]
    .dropna(subset=['value_index'])
    .sort_values('value_index', ascending=False)
    .head(10)
)
print(top_value.to_string(index=False))

Value Index Summary:
count    7721.000
mean        1.652
std         0.986
min         0.180
25%         1.050
50%         1.450
75%         2.050
max        16.000
Name: value_index, dtype: float64

Top 10 highest value restaurants:
   restaurant_name  location primary_cuisine  price_num  rating  value_index
Anitha Jowari Roti Hyderabad          Indian       30.0     4.8       16.000
    Nammane Holige Bangalore          Indian       30.0     4.5       15.000
              Bite Bangalore    Healthy Food       32.0     4.6       14.375
            Vadoji Bangalore          Snacks       40.0     4.9       12.250
      Taste N Bite   Kolkata         Burgers       40.0     4.3       10.750
            Chillz   Kolkata    North Indian       50.0     5.0       10.000
      Crispy World   Kolkata          Snacks       50.0     5.0       10.000
       City Samosa    Mumbai          Snacks       40.0     3.9        9.750
           Deshi 6   Kolkata    South Indian       50.0     4.8        9.

---
## KPI 3 — Price Band Segmentation

In [5]:
df['price_band'] = pd.cut(
    df['price_num'],
    bins=[0, 150, 300, 500, 99999],
    labels=['Budget (<Rs150)', 'Mid (Rs150-300)', 'Premium (Rs300-500)', 'Luxury (Rs500+)']
)

band_kpi = df.groupby('price_band', observed=True).agg(
    count=('restaurant_name', 'count'),
    avg_rating=('rating', 'mean'),
    avg_price=('price_num', 'mean')
).round(3)

print("Price Band Summary:")
print(band_kpi.to_string())

Price Band Summary:
                     count  avg_rating  avg_price
price_band                                       
Budget (<Rs150)       1318       4.055    127.774
Mid (Rs150-300)       5705       3.973    245.516
Premium (Rs300-500)   2242       4.087    424.731
Luxury (Rs500+)        793       4.233    795.219


---
## KPI 4 — Demand Index by City

In [6]:
df['demand_index'] = df.groupby('location')['ratings_count'].transform('mean')

city_kpi = (
    df.groupby('location')
    .agg(
        avg_demand=('ratings_count', 'mean'),
        avg_rating=('rating', 'mean'),
        restaurant_count=('restaurant_name', 'count')
    )
    .round(2)
    .sort_values('avg_demand', ascending=False)
)

print("Demand Index by City:")
print(city_kpi.to_string())

Demand Index by City:
           avg_demand  avg_rating  restaurant_count
location                                           
Bangalore     1166.09        4.13              1720
Hyderabad      998.66        3.98              1998
Delhi          845.23        3.99              1783
Mumbai         783.91        4.12              1893
Chennai        743.31        4.02               715
Kolkata        684.32        3.96              1974


---
## Additional Segmentation Columns for Tableau

In [7]:
# Rating band
df['rating_band'] = pd.cut(
    df['rating'],
    bins=[0, 3.0, 3.5, 4.0, 4.5, 5.0],
    labels=['Poor (<3.0)', 'Below Avg (3.0-3.5)', 'Average (3.5-4.0)', 'Good (4.0-4.5)', 'Excellent (4.5+)']
)

# Offer tier
df['offer_tier'] = df['num_offers'].apply(
    lambda x: 'No Offers' if x == 0 else ('Low (1-2)' if x <= 2 else 'High (3+)')
)

# Reliable rating flag — restaurants with enough reviews to be meaningful
df['reliable_rating'] = ((df['ratings_count'] >= 10) & df['rating'].notna()).astype(int)

print("Rating Band distribution:")
print(df['rating_band'].value_counts().sort_index().to_string())
print()
print("Offer Tier distribution:")
print(df['offer_tier'].value_counts().to_string())
print()
print("Reliable ratings:", df['reliable_rating'].sum(), "restaurants")

Rating Band distribution:
rating_band
Poor (<3.0)             479
Below Avg (3.0-3.5)     714
Average (3.5-4.0)      2013
Good (4.0-4.5)         3611
Excellent (4.5+)        921

Offer Tier distribution:
offer_tier
High (3+)    5013
Low (1-2)    4974
No Offers      96

Reliable ratings: 6487 restaurants


---
## Export

In [8]:
# Convert categoricals to string for CSV
for col in ['price_band', 'rating_band']:
    df[col] = df[col].astype(str).replace('nan', '')

TABLEAU_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(TABLEAU_PATH, index=False)

print(f"Saved to: {TABLEAU_PATH}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print()
print("Final columns:")
print(df.columns.tolist())

Saved to: /Users/krishiv/Documents/GitHub/B_G19_SwiggyAnalysis/data/processed/tableau_ready_dataset.csv
Shape: 10,083 rows x 19 columns

Final columns:
['restaurant_name', 'cuisine', 'rating', 'offer_name', 'area', 'pure_veg', 'location', 'price_num', 'ratings_count', 'primary_cuisine', 'is_veg', 'num_offers', 'avg_rating_by_cuisine', 'value_index', 'price_band', 'demand_index', 'rating_band', 'offer_tier', 'reliable_rating']


In [9]:
# Verification checklist
required = [
    'rating', 'price_num', 'ratings_count', 'primary_cuisine',
    'is_veg', 'num_offers', 'avg_rating_by_cuisine', 'value_index',
    'price_band', 'rating_band', 'demand_index', 'offer_tier', 'reliable_rating'
]

print("Column check:")
for col in required:
    status = 'OK' if col in df.columns else 'MISSING'
    nulls = df[col].isnull().sum() if col in df.columns else '-'
    print(f"  {col:<30} {status}   nulls: {nulls}")

Column check:
  rating                         OK   nulls: 2345
  price_num                      OK   nulls: 25
  ratings_count                  OK   nulls: 2345
  primary_cuisine                OK   nulls: 0
  is_veg                         OK   nulls: 0
  num_offers                     OK   nulls: 0
  avg_rating_by_cuisine          OK   nulls: 1
  value_index                    OK   nulls: 2362
  price_band                     OK   nulls: 25
  rating_band                    OK   nulls: 2345
  demand_index                   OK   nulls: 0
  offer_tier                     OK   nulls: 0
  reliable_rating                OK   nulls: 0
